In [1]:
# csherwood@usgs.gov, 2026-08-05, generated with Claude Sonnet 5
# ============================================================
# Summarize bulk_ndbc_compare_stats.csv (written by the plotting
# notebook) by model: mean/median Bias, RMSD, r, WSS across buoys,
# for each wave parameter (hmo/tp/wdir). Buoys with no comparison
# data (N=0, e.g. no NDBC obs or no model output) are excluded from
# the averages but counted separately.
#
# Usage: python summarize_stats_by_model.py [path_to_stats_csv]
# ============================================================
import sys
import pandas as pd
from pathlib import Path

STATS_CSV = "F:/crs/proj/2025_NOPP_comparison/helene_figs/bulk_ndbc_compare_stats.csv"
SUMMARY_CSV = "F:/crs/proj/2025_NOPP_comparison/helene_figs/bulk_ndbc_compare_stats"+"_by_model.csv"

VAR_ORDER = {"hmo": 0, "tp": 1, "wdir": 2}
MODEL_ORDER = {"HURRYWAVE": 0, "COAWST": 1, "ADCIRC": 2}

df = pd.read_csv(STATS_CSV)
# N=0 means there was no actual comparison (no NDBC obs / no model output at
# that buoy) -- exclude those. Keep everything else even if e.g. "r" is NaN
# for one row (undefined when a series has zero variance); pandas' mean/median
# skip individual NaNs per column rather than dropping the whole row.
df_valid = df[df["N"] > 0]

summary = (
    df_valid.groupby(["variable", "model"])
    .agg(
        n_buoys=("station_id", "nunique"),
        N_total=("N", "sum"),
        Bias_mean=("Bias", "mean"),
        Bias_median=("Bias", "median"),
        RMSD_mean=("RMSD", "mean"),
        RMSD_median=("RMSD", "median"),
        r_mean=("r", "mean"),
        r_median=("r", "median"),
        WSS_mean=("WSS", "mean"),
        WSS_median=("WSS", "median"),
    )
    .reset_index()
)
summary = (summary
           .assign(_vo=summary["variable"].map(VAR_ORDER), _mo=summary["model"].map(MODEL_ORDER))
           .sort_values(["_vo", "_mo"])
           .drop(columns=["_vo", "_mo"])
           .reset_index(drop=True))

n_excluded = len(df) - len(df_valid)
print(f"Read {len(df)} rows from {STATS_CSV}")
print(f"({n_excluded} station/variable/model rows excluded: no comparison data)\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

summary.to_csv(SUMMARY_CSV, index=False)
print(f"\nWrote model summary: {SUMMARY_CSV}")


Read 63 rows from F:/crs/proj/2025_NOPP_comparison/helene_figs/bulk_ndbc_compare_stats.csv
(15 station/variable/model rows excluded: no comparison data)

variable     model  n_buoys  N_total  Bias_mean  Bias_median  RMSD_mean  RMSD_median  r_mean  r_median  WSS_mean  WSS_median
     hmo HURRYWAVE        3      270      0.154        0.138      0.526        0.516   0.984     0.986     0.785       0.866
     hmo    COAWST        7      650      0.006        0.066      0.418        0.374   0.965     0.972     0.880       0.913
     hmo    ADCIRC        7      650      0.034        0.120      0.624        0.630   0.918     0.955     0.725       0.806
      tp HURRYWAVE        3      254     -0.001        0.247      1.681        1.832   0.848     0.817     0.690       0.622
      tp    COAWST        7      634      0.497        0.272      1.786        1.482   0.868     0.910     0.570       0.750
      tp    ADCIRC        7      634     -0.284       -0.356      2.165        1.971   0.828    